# Quick, Draw! × Fashion-MNIST — CNN Training Guide

Pipeline:
1. Download Quick Draw `.npy` files (fashion classes)
2. Merge với Fashion-MNIST
3. Train CNN
4. Export ONNX cho app vẽ tay

## 0. Install dependencies

In [ ]:
!pip install tensorflow tf2onnx onnx requests numpy matplotlib scikit-learn

## 1. Download Quick Draw dataset

Quick Draw cung cấp bitmap 28×28 dạng `.npy` — mỗi file là 1 class.

**Các class fashion có trong Quick Draw:**
| Class | URL key |
|---|---|
| T-shirt | `t-shirt` |
| Shoe | `shoe` |
| Pants | `pants` |
| Jacket | `jacket` |
| Shorts | `shorts` |
| Sock | `sock` |
| Hat | `hat` |
| Handbag | `handbag` |
| Backpack | `backpack` |
| Dress | `dress` |

In [ ]:
import os, requests, numpy as np
from pathlib import Path

# ── Class mapping: tên hiển thị → Quick Draw URL key
# Chọn 10 class tương ứng gần nhất với Fashion-MNIST
QD_CLASSES = {
    'T-shirt':  't-shirt',
    'Pants':    'pants',
    'Jacket':   'jacket',
    'Dress':    'dress',
    'Coat':     'jacket',      # Quick Draw không có 'coat', dùng jacket
    'Sandal':   'shoe',
    'Shirt':    't-shirt',
    'Sneaker':  'shoe',
    'Handbag':  'handbag',
    'Boot':     'shoe',
}

# Class cuối cùng sau khi deduplicate URL
DOWNLOAD_KEYS = list(set(QD_CLASSES.values()))
print('Unique files to download:', DOWNLOAD_KEYS)

BASE_URL  = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'
DATA_DIR  = Path('quickdraw_data')
DATA_DIR.mkdir(exist_ok=True)

def download_class(key: str, force=False):
    fname = DATA_DIR / f'{key}.npy'
    if fname.exists() and not force:
        print(f'  [skip] {key}.npy already exists')
        return
    url = BASE_URL + key.replace(' ', '%20') + '.npy'
    print(f'  Downloading {key}...', end=' ')
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(fname, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    size_mb = fname.stat().st_size / 1e6
    print(f'done ({size_mb:.1f} MB)')

for key in DOWNLOAD_KEYS:
    download_class(key)

## 2. Load và preview dữ liệu

In [ ]:
import matplotlib.pyplot as plt

# Load 1 class để xem
sample = np.load(DATA_DIR / 'shoe.npy')  # shape: (N, 784)
print(f'shoe.npy shape: {sample.shape}')
print(f'dtype: {sample.dtype}, min: {sample.min()}, max: {sample.max()}')

# Visualize 10 samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, img in zip(axes.flat, sample[:10]):
    ax.imshow(img.reshape(28, 28), cmap='gray_r')  # gray_r: đen nền trắng
    ax.axis('off')
plt.suptitle('Quick Draw — shoe samples', fontsize=14)
plt.tight_layout()
plt.show()

# So sánh với Fashion-MNIST
from tensorflow.keras.datasets import fashion_mnist
(fm_train, fm_y_train), (fm_test, fm_y_test) = fashion_mnist.load_data()

# Sneaker class = label 7
sneakers = fm_train[fm_y_train == 7][:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, img in zip(axes.flat, sneakers):
    ax.imshow(img, cmap='gray_r')
    ax.axis('off')
plt.suptitle('Fashion-MNIST — sneaker samples', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build dataset

**Chiến lược:** Dùng Quick Draw làm nguồn chính (vì app vẽ tay), Mix thêm Fashion-MNIST để model biết ảnh "chuẩn" cũng.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Class definitions (10 class, mỗi class map sang 1 hoặc nhiều QD keys)
CLASS_DEFS = [
    ('T-shirt/top', ['t-shirt'],  0),   # (tên hiển thị, qd_keys, fashion_mnist_label)
    ('Trouser',     ['pants'],    1),
    ('Pullover',    ['jacket'],   2),
    ('Dress',       ['dress'],    3),
    ('Coat',        ['jacket'],   4),
    ('Sandal',      ['shoe'],     5),
    ('Shirt',       ['t-shirt'],  6),
    ('Sneaker',     ['shoe'],     7),
    ('Bag',         ['handbag'],  8),
    ('Ankle boot',  ['shoe'],     9),
]

SAMPLES_PER_CLASS_QD = 10_000   # Quick Draw samples per class
SAMPLES_PER_CLASS_FM = 2_000    # Fashion-MNIST samples per class (optional mix)
USE_FASHION_MNIST_MIX = True    # bật/tắt mix Fashion-MNIST

X_list, y_list = [], []

print('Loading Quick Draw data...')
for class_idx, (class_name, qd_keys, fm_label) in enumerate(CLASS_DEFS):
    imgs_for_class = []
    
    # Load từ Quick Draw
    for key in qd_keys:
        path = DATA_DIR / f'{key}.npy'
        if not path.exists():
            print(f'  WARNING: {path} not found, skipping')
            continue
        data = np.load(path)                           # (N, 784) uint8
        data = data[:SAMPLES_PER_CLASS_QD // len(qd_keys)]
        imgs_for_class.append(data)
    
    if imgs_for_class:
        combined = np.concatenate(imgs_for_class, axis=0)
        X_list.append(combined)
        y_list.append(np.full(len(combined), class_idx))
        print(f'  [{class_idx}] {class_name}: {len(combined)} QD samples')
    
    # Mix Fashion-MNIST
    if USE_FASHION_MNIST_MIX:
        fm_imgs = fm_train[fm_y_train == fm_label]
        fm_imgs = fm_imgs[:SAMPLES_PER_CLASS_FM].reshape(-1, 784)
        # Fashion-MNIST: nền đen nét trắng → cần invert để giống Quick Draw
        # Quick Draw: nền trắng nét đen → invert FM
        fm_imgs = 255 - fm_imgs
        X_list.append(fm_imgs)
        y_list.append(np.full(len(fm_imgs), class_idx))
        print(f'       {class_name}: +{len(fm_imgs)} FM samples')

X = np.concatenate(X_list, axis=0).astype(np.float32) / 255.0  # normalize 0-1
y = np.concatenate(y_list, axis=0)

print(f'\nTotal dataset: X={X.shape}, y={y.shape}')
print(f'Class distribution: {np.bincount(y)}')

In [ ]:
from tensorflow.keras.utils import to_categorical

# ── Reshape cho CNN: (N, 28, 28, 1)
X = X.reshape(-1, 28, 28, 1)

# ── Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

y_train_cat = to_categorical(y_train, 10)
y_val_cat   = to_categorical(y_val,   10)
y_test_cat  = to_categorical(y_test,  10)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 4. Data Augmentation

Quick Draw sketches rất đa dạng về scale/rotation nên augmentation giúp nhiều.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import RandomRotation, RandomZoom, RandomTranslation

# Augmentation pipeline (chỉ apply lúc training)
augment = tf.keras.Sequential([
    RandomRotation(factor=0.15),           # ±15 độ
    RandomZoom(height_factor=(-0.1, 0.1)), # zoom in/out 10%
    RandomTranslation(height_factor=0.1, width_factor=0.1),  # shift 10%
], name='augmentation')

# Preview augmentation
sample_img = X_train[:9]
aug_imgs   = augment(sample_img, training=True).numpy()

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
class_names = [c[0] for c in CLASS_DEFS]
for i in range(9):
    axes[i//3, (i%3)*2].imshow(sample_img[i].squeeze(), cmap='gray_r')
    axes[i//3, (i%3)*2].set_title(f'orig: {class_names[y_train[i]]}', fontsize=8)
    axes[i//3, (i%3)*2].axis('off')
    axes[i//3, (i%3)*2+1].imshow(aug_imgs[i].squeeze(), cmap='gray_r')
    axes[i//3, (i%3)*2+1].set_title('augmented', fontsize=8)
    axes[i//3, (i%3)*2+1].axis('off')
plt.tight_layout()
plt.show()

## 5. Build CNN Model

Dùng **MobileNetV2-style depthwise separable conv** — nhẹ hơn CNN thuần mà accuracy tốt hơn.

In [ ]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import (
    Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, Flatten, Activation
)

def build_cnn(num_classes=10, input_shape=(28, 28, 1)):
    model = Sequential([
        Input(shape=input_shape),
        
        # ── Block 1
        Conv2D(32, 3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(32, 3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(2),               # 14×14
        Dropout(0.25),
        
        # ── Block 2 (Separable conv: nhẹ hơn, tốt hơn với sketch)
        SeparableConv2D(64, 3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        SeparableConv2D(64, 3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(2),               # 7×7
        Dropout(0.25),
        
        # ── Block 3
        SeparableConv2D(128, 3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        GlobalAveragePooling2D(),      # thay Flatten: ít param hơn, ít overfit hơn
        
        # ── Classifier
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ], name='QuickDraw_CNN')
    return model

model = build_cnn()
model.summary()

## 6. Train

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        'best_quickdraw_cnn.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

# ── Dataset với augmentation (chỉ train set)
BATCH_SIZE = 256

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
    .shuffle(10_000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (augment(x, training=True), y),
         num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
)

## 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

# ── Test accuracy
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

# ── Classification report
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
class_names = [c[0] for c in CLASS_DEFS]
print('\n', classification_report(y_test, y_pred, target_names=class_names))

# ── Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

## 8. Export ONNX

In [ ]:
import tf2onnx, onnx, tensorflow as tf

# Load best model
best_model = tf.keras.models.load_model('best_quickdraw_cnn.keras')

# Export
input_sig  = [tf.TensorSpec((None, 28, 28, 1), tf.float32, name='inputs')]
model_proto, _ = tf2onnx.convert.from_keras(best_model, input_signature=input_sig)
onnx.save(model_proto, 'fashion_mnist_cnn.onnx')
print('Saved: fashion_mnist_cnn.onnx')

# ── Verify ONNX
import onnxruntime as ort
sess       = ort.InferenceSession('fashion_mnist_cnn.onnx')
inp_name   = sess.get_inputs()[0].name
inp_shape  = sess.get_inputs()[0].shape
print(f'ONNX input: {inp_name} {inp_shape}')

dummy = np.random.rand(1, 28, 28, 1).astype(np.float32)
out   = sess.run(None, {inp_name: dummy})[0]
print(f'Output shape: {out.shape}  sum={out.sum():.4f}  (should be ~1.0)')
print('✓ ONNX export OK — copy file vào folder app rồi chạy!')

## 9. Quick test giống app vẽ tay

Simulate pipeline giống hệt `preprocess()` trong app để verify.

In [ ]:
import cv2

def simulate_app_preprocess(img_28x28: np.ndarray) -> np.ndarray:
    """Giống hệt preprocess() trong app — verify pipeline khớp."""
    # img_28x28: Quick Draw format (uint8, nền trắng nét đen, 0-255)
    arr = img_28x28.astype(np.float32)
    # Invert để tìm bbox
    inv = 255.0 - arr
    rows = np.any(inv > 20, axis=0)
    cols = np.any(inv > 20, axis=1)
    if rows.any() and cols.any():
        rmin, rmax = np.where(cols)[0][[0, -1]]
        cmin, cmax = np.where(rows)[0][[0, -1]]
        cropped = inv[rmin:rmax+1, cmin:cmax+1]
    else:
        cropped = inv
    h, w  = cropped.shape
    side  = max(h, w)
    pad   = max(1, int(side * 0.15))
    sq    = np.zeros((side+2*pad, side+2*pad), dtype=np.float32)
    sq[pad+(side-h)//2:pad+(side-h)//2+h,
       pad+(side-w)//2:pad+(side-w)//2+w] = cropped
    resized = cv2.resize(sq, (28, 28), interpolation=cv2.INTER_AREA)
    out = resized / 255.0
    out = cv2.GaussianBlur(out, (3, 3), 0)
    return out.reshape(1, 28, 28, 1).astype(np.float32)

# Test trên 20 samples ngẫu nhiên
indices  = np.random.randint(0, len(X_test), 20)
correct  = 0
fig, axes = plt.subplots(4, 5, figsize=(14, 12))

for ax, idx in zip(axes.flat, indices):
    img_raw   = (X_test[idx].squeeze() * 255).astype(np.uint8)
    processed = simulate_app_preprocess(img_raw)
    probs     = sess.run(None, {inp_name: processed})[0][0]
    pred      = probs.argmax()
    true_lbl  = y_test[idx]
    color     = 'green' if pred == true_lbl else 'red'
    correct  += (pred == true_lbl)
    
    ax.imshow(img_raw, cmap='gray_r')
    ax.set_title(
        f'True: {class_names[true_lbl]}\n'
        f'Pred: {class_names[pred]} ({probs[pred]*100:.0f}%)',
        color=color, fontsize=8
    )
    ax.axis('off')

plt.suptitle(f'App pipeline accuracy on 20 samples: {correct}/20', fontsize=13)
plt.tight_layout()
plt.show()